In [1]:
import os
import sys

PROJECT_PATH = r"C:\Mestrado\Graph_Pruning\OpenGraph\node_classification"

os.chdir(PROJECT_PATH)

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

print(os.getcwd())

C:\Mestrado\Graph_Pruning\OpenGraph\node_classification


In [11]:
def evaluate_model(exp, times=10):

    all_results = {}

    for handler in exp.multi_handler.tst_handlers:

        res_summary = {}

        for i in range(times):

            reses = exp.test_epoch(
                handler.tst_loader,
                handler
            )

            exp.add_res_to_summary(
                res_summary,
                reses
            )

            exp.multi_handler.remake_initial_projections()

        for key in res_summary:
            res_summary[key] /= times

        all_results[
            handler.data_name
        ] = res_summary

    return all_results

In [3]:
import torch as t
import numpy as np
import pandas as pd
import pickle

import Utils.TimeLogger as logger
from Utils.TimeLogger import log

from data_handler import *
from model import *
from main import *

In [4]:
args.gpu = '0'

datasets = [
    'cora',
    'pubmed',
    'citeseer'
]

args.load_model = None

In [5]:
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu

if len(args.gpu.split(',')) > 1:
    args.devices = ['cuda:0', 'cuda:1']
else:
    args.devices = ['cuda:0', 'cuda:0']

args.devices = [
    t.device(device)
    for device in args.devices
]

print("Devices:", args.devices)

Devices: [device(type='cuda', index=0), device(type='cuda', index=0)]


In [8]:
trn_datasets = [
    "cora",
    "pubmed",
    "citeseer"
]

tst_datasets = [
    "cora",
    "pubmed",
    "citeseer"
]

multi_handler = MultiDataHandler(
    trn_datasets,
    tst_datasets
)

print("Train datasets:", trn_datasets)
print("Test datasets:", tst_datasets)

Dataset: citeseer, Node num: 3333, Edge num: 10344
Dataset: pubmed, Node num: 19720, Edge num: 89768
Dataset: cora, Node num: 2715, Edge num: 11836
Train datasets: ['cora', 'pubmed', 'citeseer']
Test datasets: ['cora', 'pubmed', 'citeseer']


In [9]:
exp = Exp(multi_handler)

exp.prepare_model()

log("Model Prepared")

Total params: 25.1904
Trainable params: 25.1904
Non-trainable params: 0.0
2026-09-02 18:45:19.476153: Model Prepared


In [10]:
args.load_model = "pretrn_gen1"

exp.load_model()

print(exp.model)

Loading model from: C:\Mestrado\Graph_Pruning\OpenGraph\Models\pretrn_gen1.mod
Loading history from: C:\Mestrado\Graph_Pruning\OpenGraph\History\pretrn_gen1.his
2026-09-02 18:45:43.233330: Model Loaded
OpenGraph(
  (topoEncoder): TopoEncoder(
    (layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=False)
  )
  (graphTransformer): GraphTransformer(
    (gt_layers): Sequential(
      (0): GTLayer(
        (multi_head_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias=False)
        )
        (dense_layers): Sequential(
          (0): FeedForwardLayer(
            (linear): Linear(in_features=1024, out_features=1024, bias=True)
            (act): LeakyReLU(negative_slope=0.5)
          )
          (1): FeedForwardLayer(
            (linear): Linear(in_features=1024, out_features=1024, bias=True)
            (act): LeakyReLU(negative_slope=0.5)
          )
        )
        (layer_norm1): LayerNorm((102

In [ ]:
#exemplo de como aplicar o pruning, só chamar o seu metodo de polda, salvar o modelo 
# e depois carregar ele para testar

#pruner = GlobalMagnitudePruner(exp.model)

#exp.model = pruner.prune(0.5)

In [12]:
baseline_results = evaluate_model(
    exp,
    times=10
)

baseline_results

{'citeseer': {'Acc': 0.0819, 'F1': 0.07855282582284608},
 'pubmed': {'Acc': 0.08320000000000001, 'F1': 0.05778084474225324},
 'cora': {'Acc': 0.7505, 'F1': 0.7430188392930781}}